# 🧠 Building a Single-Agent Task Router

## Overview
This notebook builds a **Single-Agent Smart Assistant** that can:
- Take in a user's query
- Figure out which kind of task it is
- Hand it off to the correct tool
- Send back a clean, structured JSON response

### Tasks the agent covers:
- Arithmetic requests → Math Tool
- Topic/keyword requests → Topic Tool
- Anything else → Default Reply

---
### 🛠️ Core Components
- Decision-making logic for the agent
- Conditional routing between tools
- The tools themselves
- Basic error handling so a bad query never crashes the pipeline

### 🚀 Extras Added
- Smarter routing order
- Logging of each decision the agent makes
- A third tool beyond the two required ones

In [ ]:
import json
import re

In [ ]:
# 🛠️ TOOL 1: Math Solver

def solve_math(expression: str):
    """Evaluate a simple arithmetic expression."""
    expression = expression.replace("x", "*").replace("X", "*")
    if not re.fullmatch(r"[0-9\s\.\+\-\*\/\(\)]+", expression):
        return "Invalid expression"
    try:
        return eval(expression, {"__builtins__": {}}, {})
    except ZeroDivisionError:
        return "Cannot divide by zero"
    except Exception:
        return "Invalid expression"

In [ ]:
# 🛠️ TOOL 2: Topic Finder

def find_topics(text: str, limit: int = 5):
    """Pull out the likely topic words from a sentence."""
    words = re.findall(r"[A-Za-z']+", text)
    found = []
    for w in words:
        w_lower = w.lower()
        if len(w_lower) > 4 and w_lower not in found:
            found.append(w_lower)
    return found[:limit]

In [ ]:
# 🛠️ TOOL 3 (BONUS): Text Stats

def text_stats(text: str):
    """Return basic word/character counts for a piece of text."""
    words = text.split()
    return {"word_count": len(words), "char_count": len(text)}

## 🤖 Agent Decision Logic

👉 How queries get routed:
- Contains "solve" → Math Tool
- Contains "topics" → Topic Tool
- Contains "stats" → Text Stats Tool (bonus)
- Anything else → Default Reply

In [ ]:
# 🤖 AGENT FUNCTION

def agent(query: str) -> dict:
    """
    Single-Agent Smart Assistant
    """

    print(f"[LOG] Query received: {query}")

    lowered = query.lower()

    try:

        # -------------------------------
        # Math Routing
        # -------------------------------
        if "solve" in lowered:

            print("[LOG] Routing -> Math Tool")

            expression = lowered.replace("solve", "").strip()
            result = solve_math(expression)

            return {
                "type": "math",
                "result": result
            }

        # -------------------------------
        # Topic Routing
        # -------------------------------
        elif "topics" in lowered:

            print("[LOG] Routing -> Topic Tool")

            text = re.sub(r"topics (in|for|from)?", "", query, flags=re.IGNORECASE).strip()
            result = find_topics(text)

            return {
                "type": "topics",
                "result": result
            }

        # -------------------------------
        # Text Stats Routing (bonus)
        # -------------------------------
        elif "stats" in lowered:

            print("[LOG] Routing -> Text Stats Tool")

            text = re.sub(r"stats (for|on)?", "", query, flags=re.IGNORECASE).strip()
            result = text_stats(text)

            return {
                "type": "stats",
                "result": result
            }

        # -------------------------------
        # Default Reply
        # -------------------------------
        else:

            print("[LOG] Routing -> Default Reply")

            return {
                "type": "general",
                "result": "I can solve math, find topics, or give text stats. "
                          "Try including 'solve', 'topics', or 'stats' in your query."
            }

    except Exception as e:

        print("[LOG] Error:", e)

        return {
            "type": "error",
            "result": str(e)
        }

## 📦 Response Format

```
{
  "type": "math / topics / stats / general / error",
  "result": ...
}
```

In [ ]:
# 🧪 Test Cases

queries = [
    "solve 12 * (3 + 4)",
    "topics in Renewable energy is reshaping global electricity markets",
    "What is machine learning?",
    "stats for The old lighthouse stood quietly against the storm",
    "solve 10 / 0"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))